### **Introduction**
This notebook shows how to stitch two overlapping volumes using the code developed in the source code.

In the T-rex scans, the individual reconstructions are not aligned only by translation, as the specimen had to be manually moved around, back and forth, and fit inside the FOV of the scanner.

Inter-scan misalignment includes:
- 3D rotations
- Axis flips
- Small isotropic scaling differences

Therefore, stitching requires the use of rigid and similarity transforms.

The registration uses a mutual information metric to optimize the alignment. Thus the alignment maximizes the statistical dependence between intensities of voxels mapped to the same spatial location.

Masking was used on some of the volumes in the pipeline to guide the optimizer to only look at the overlapping regions. This is because some of the scans had very little overlap, which made them more difficult to stitch.

### **Imports**

In [2]:
import os
import matplotlib.pyplot as plt
import qim3d.viz

In [3]:
from src import cfg
from scripts.reconstruct import reconstruct_scans
from scripts.downsample_recons import downsample_recons
from src.io import load_recon
from src.volume_stitcher import estimate_transform, stitch

### **Reconstruction the scans**

Available is two scans diagonal from each other in layer 4.

In [5]:
scans = [cfg.SCANS_BY_LAYER[4][2], cfg.SCANS_BY_LAYER[4][0]]
print(scans)

['top_4_3-ny', 'top_4_1']


We reconstruct the scans using the padding method described in the `padding.ipynb` notebook.

In [ ]:
reconstruct_scans(scans=scans, dataset=cfg.full_res)

Due to the stitching being quite time consuming on the full scans, we here downsample the reconstructions and use those instead.

In [6]:
downsample_recons(scans=scans)

Wrote recon: /work3/s214743/ct_data/casper/recons/padded_downsampled/top_4_3-ny.npy with shape (334, 334, 334)
Wrote recon: /work3/s214743/ct_data/casper/recons/padded_downsampled/top_4_1.npy with shape (334, 334, 334)


### **Estimating the transform**

In [ ]:
vol_fixed = load_recon(scans[0], dataset=cfg.downsampled)
vol_moving = load_recon(scans[1], dataset=cfg.downsampled)

In [ ]:
A_hom = estimate_transform(vol_fixed, vol_moving, type_of_transform='Rigid')

The output is a 4x4 homogeneous matrix representing the transform:

In [ ]:
print(A_hom)

### **Stitching the volumes**

Now the stitching is done to combine the two volumes together into one larger volume. We also include some more outputs with `debug=True` to visualize what is going.

In [ ]:
vol_stitched, canvas1, canvas2 = stitch(vol_fixed, vol_moving, A_hom, debug=True)

See the alignment by sliding between each original volume transformed to the same coordinate system:

In [ ]:
qim3d.viz.overlay(canvas1, canvas2)

See the resulting stitch:

In [ ]:
qim3d.viz.slicer(vol_stitched)